# Teacher-Student Alignment Evaluation

For each distilled student (KL-KD, Fixed-OT-KD, Adaptive-OT-KD, plus the no-KD baseline), this measures how closely it tracks the teacher on the test set:

| Metric | What it captures |
|---|---|
| **KL(p_T \|\| p_S)** | divergence on softened class distributions (lower = closer) |
| **Sinkhorn W_ε** | entropy-regularized Wasserstein distance with uniform cost |
| **cos(z_T, z_S)** | per-sample cosine similarity of raw logits, averaged |
| **top-1 agreement** | fraction of test samples where argmax matches the teacher |
| **top-5 agreement** | fraction where the teacher's top-1 sits in the student's top-5 |

Runtime is fast: ~10 s per dataset on a T4 GPU, ~2 min on CPU. No big downloads — uses the standard CIFAR-10/100 test sets that torchvision auto-fetches.

## 0. GPU runtime check

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > GPU'

## 1. Get the project into Colab
Pick **one** of 1A or 1B.

### 1A. (recommended) Clone from GitHub
Clones from https://github.com/ayman-tech/sinkhorn-vision-kd

In [ ]:
%cd /content
!rm -rf sinkhorn-vision-kd
!git clone --depth 1 https://github.com/ayman-tech/sinkhorn-vision-kd.git sinkhorn-vision-kd
%cd sinkhorn-vision-kd
import os; print(sorted(os.listdir('.')))

### 1B. Upload a project zip

In [ ]:
%cd /content
from google.colab import files
import os, zipfile
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn) as zf:
            zf.extractall('./')
        os.remove(fn)
if os.path.isdir('sinkhorn-vision-kd'):
    %cd sinkhorn-vision-kd
print(sorted(os.listdir('.')))

## 2. Sanity check — checkpoints present?

In [ ]:
import os
all_ok = True
for ds in ['cifar10', 'cifar100']:
    d = f'./checkpoints/{ds}'
    print(f'\n[{ds}]  ({d})')
    if not os.path.isdir(d):
        print('   MISSING DIR'); all_ok = False; continue
    needed = [
        f'{ds}_resnet110_teacher.pth',
        'resnet20_no_kd_best.pth',
        'kl_kd_best.pth',
        'sinkhorn_kd_best.pth',
        'adaptive_sinkhorn_kd_best.pth',
    ]
    for f in needed:
        p = os.path.join(d, f)
        print(('   OK   ' if os.path.exists(p) else '   MISS '), f)
        if not os.path.exists(p): all_ok = False
print('\nALL GOOD' if all_ok else '\nSOME CHECKPOINTS MISSING — see above')

## 3. Alignment — CIFAR-10

In [ ]:
!python evaluate_alignment.py \
    --dataset cifar10 \
    --batch_size 512 \
    --num_workers 2 \
    --save_csv alignment_cifar10.csv

## 4. Alignment — CIFAR-100

In [ ]:
!python evaluate_alignment.py \
    --dataset cifar100 \
    --batch_size 512 \
    --num_workers 2 \
    --save_csv alignment_cifar100.csv

## 5. Pretty-print the CSVs

In [ ]:
import os, pandas as pd
for f in ['alignment_cifar10.csv', 'alignment_cifar100.csv']:
    if not os.path.exists(f):
        continue
    print(f'\n=== {f} ===')
    df = pd.read_csv(f)
    print(df.round(4).to_string(index=False))

## 6. Plot the metrics side-by-side

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt

metrics = [('kl', 'KL(T||S) — lower better'),
           ('wasserstein', 'Sinkhorn W_eps — lower better'),
           ('cosine', 'cos(z_T, z_S) — higher better'),
           ('top1_agree', 'top-1 agreement % — higher better')]

for ds in ['cifar10', 'cifar100']:
    csv = f'alignment_{ds}.csv'
    if not os.path.exists(csv):
        continue
    df = pd.read_csv(csv)
    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4))
    for ax, (col, title) in zip(axes, metrics):
        ax.bar(df.method, df[col])
        ax.set_title(f'{ds} — {title}')
        ax.tick_params(axis='x', rotation=30)
        ax.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

## 7. Download CSVs

In [ ]:
from google.colab import files
import os
for f in ['alignment_cifar10.csv', 'alignment_cifar100.csv']:
    if os.path.exists(f):
        files.download(f)